# 60 · AI & RAG — end-to-end retrieval-augmented generation

**This is the whole loop, wired end to end against live services: retrieve → augment →
generate.** A language model knows only what was in its training data; it has never seen this
lab's docs, its backlog, or its runbooks. RAG closes that gap without retraining: we **embed**
the user's question into the same vector space as a corpus, **retrieve** the nearest chunks
from that corpus, **augment** a prompt with them as context, and let the model **generate** an
answer *grounded in text it was handed* rather than in its parameters. The retrieval half is
notebook `30`'s Qdrant search; the generation half is the LiteLLM gateway (notebook `61`); this
notebook is the two halves joined into one working pipeline over the lab's own knowledge.

> **The one rule that makes retrieval work: the query and the corpus must be embedded by the
> SAME model.** A vector search compares directions in a shared space; embed the question with a
> different model and the numbers are meaningless, the "nearest" chunks are noise, and the answer
> is grounded in garbage. So we load the exact model the corpus was built with — `BAAI/bge-base-en-v1.5`
> — and embed the query with it, the same way the platform's own retriever service does.

### What this notebook does

It embeds a real question about the platform, retrieves the top-k chunks from the
`weyland_chunks` corpus in Qdrant, **shows those chunks** (source + snippet + score) so grounding
is visible, stuffs them into a grounded prompt, and generates an answer through the LiteLLM
gateway's `wl-rag` lane. Then it asks the **same question with no context** for contrast — so you
can see exactly what the retrieval added.

> **Read-only on the stores.** Every Qdrant call here is a `query_points` — retrieval only.
> Nothing is upserted, deleted, or re-indexed; the corpus is read exactly as the hydration
> pipeline left it, so — as in notebooks `22`/`30` — there is **no cleanup section**. The only
> thing that runs is a search and a generation.

> **Implementation note.** This wires the loop **explicitly** — `qdrant-client` for retrieval and
> the `openai` client (LiteLLM is OpenAI-compatible) for generation — rather than through a
> framework. That is deliberate: the platform's own corpus stores chunk text under `content`
> (not a framework's expected `text`/`_node_content` blob), so a drop-in `QdrantVectorStore` can't
> reconstruct nodes from it — the same reason `weyland-agent/retrievers.py` wraps raw queries. The
> explicit path is a dozen honest lines, has no version pins to fight the base image, and shows
> every step of RAG with nothing hidden.

## Setup

Three libraries are **not** in the singleuser base image (which ships `polars`, `s3fs`,
`pyarrow`, `duckdb`, `fastavro`), so we install them here:

- **`sentence-transformers`** — to load `BAAI/bge-base-en-v1.5` and embed the query locally. On
  first run it downloads the model (~440MB) from Hugging Face; the singleuser pod has public
  egress and the HF cache lives under the PVC home, so the download persists across restarts and
  only happens once. CPU is fine — we embed a single short query.
- **`qdrant-client`** — the vector-store client, exactly as in notebook `30`.
- **`openai`** — the generation client; the LiteLLM gateway is OpenAI-compatible, so the stock
  `openai` SDK talks to it unchanged.

`polars` — used to render the retrieved-context frame, as in notebooks `22`/`30` — already ships
in the image.

In [1]:
%pip install -q sentence-transformers qdrant-client openai

Note: you may need to restart the kernel to use updated packages.


## Connect — the vector store and the LLM gateway

Both connections are **env-driven**, the same pattern the query and vector notebooks use. The
committed defaults are the **in-cluster** service URLs; a validation run overrides them via the
environment (e.g. to a NodePort) **without editing the notebook**, so the notebook never captures
a resolved address — each connection is proven by **what it returns**, not by echoing an endpoint.

- **Qdrant** (`QDRANT_URL`, default `qdrant.weyland.svc.cluster.local:6333`) — unmeshed and
  unauthenticated on the LAN, so the client needs only a URL, no API key.
- **LiteLLM** (`LITELLM_BASE_URL`, default `litellm.weyland.svc.cluster.local:4000/v1`) — the
  OpenAI-compatible egress gateway. It **does** require auth: a bearer token read from
  `LITELLM_MASTER_KEY` (never committed — in the singleuser pod it is injected from the
  `litellm-creds` secret). We use the model alias **`wl-rag`**, the gateway's retrieval-grounded
  generation lane.

In [2]:
import os
from qdrant_client import QdrantClient
from openai import OpenAI
import polars as pl

# committed defaults = in-cluster DNS; a validation run overrides these via env, never in the notebook
QDRANT_URL       = os.environ.get("QDRANT_URL", "http://qdrant.weyland.svc.cluster.local:6333")
LITELLM_BASE_URL = os.environ.get("LITELLM_BASE_URL", "http://litellm.weyland.svc.cluster.local:4000/v1")
LITELLM_KEY      = os.environ["LITELLM_MASTER_KEY"]   # required; injected from the litellm-creds secret in-pod

COLLECTION = "weyland_chunks"   # the RAG corpus (see notebook 30)
GEN_MODEL  = "wl-rag"           # LiteLLM's retrieval-grounded generation lane

qdrant = QdrantClient(url=QDRANT_URL)
llm    = OpenAI(base_url=LITELLM_BASE_URL, api_key=LITELLM_KEY)

def snippet(text, n=110):   # flatten whitespace + truncate long chunk text for display
    return None if text is None else " ".join(str(text).split())[:n]

# Prove each connection by WHAT IT RETURNS, never by echoing a resolved endpoint.
corpus = qdrant.get_collection(COLLECTION)
print(f"Qdrant  - collection {COLLECTION!r}: "
      f"{corpus.points_count} points, {corpus.config.params.vectors.size}-dim, "
      f"{getattr(corpus.config.params.vectors.distance, 'value', corpus.config.params.vectors.distance)}")
print(f"LiteLLM - gateway reachable, {len(llm.models.list().data)} models exposed; generating with {GEN_MODEL!r}")

Qdrant  - collection 'weyland_chunks': 8378 points, 768-dim, Cosine


LiteLLM - gateway reachable, 235 models exposed; generating with 'wl-rag'


## Retrieve, step 1 — embed the query with the corpus's own model

The corpus was embedded with **`BAAI/bge-base-en-v1.5`** (768-dim, cosine) by the hydration
pipeline — `weyland_pipeline/resources/sentence_transformer.py` calls
`model.encode(text, normalize_embeddings=True)` on each chunk. To land in the same vector space
we must embed the query the **same way**, and two details matter:

- **The same model.** Loaded below — no substitutes, or the vectors don't compare.
- **`normalize_embeddings=True`.** The corpus vectors are L2-normalized, so we normalize the
  query too; with cosine distance that keeps scores directly comparable.
- **No query prefix.** bge-v1.5 *documents* an optional instruction prefix
  (`"Represent this sentence for searching relevant passages:"`) for the query side — but **this
  corpus was indexed with bare text and no prefix**, and the platform's live retriever
  (`weyland-agent/retrievers.py`) embeds the query bare too (`get_text_embedding(query)`, the
  passage path). Query and corpus must be embedded *identically*; since the corpus is bare, the
  query is bare. Matching the repo beats matching the model card here.

The first run downloads the model; later runs load it from the PVC-backed HF cache in seconds.

In [3]:
from sentence_transformers import SentenceTransformer

# the EXACT model the corpus was built with (qdrant_write.py DIMS=768; sentence_transformer.py model_name)
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

question = "How does the lab do GitOps deployment?"

# bare text + normalize_embeddings=True == exactly how the corpus + the live retriever embed. No prefix.
query_vec = embedder.encode(question, normalize_embeddings=True).tolist()
print(f"question   : {question}")
print(f"query vector: {len(query_vec)} dims, L2-normalized (cosine space) - matches the {COLLECTION!r} geometry")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

question   : How does the lab do GitOps deployment?
query vector: 768 dims, L2-normalized (cosine space) - matches the 'weyland_chunks' geometry


## Retrieve, step 2 — the top-k nearest chunks

Now the vector search: hand the query vector to Qdrant and get back the points whose vectors are
nearest — the passages closest *in meaning* to the question, no keyword match required. We pull
the payload back with each hit so we get the real chunk text (`content`) and its provenance
(`source_name`), and we render them as a polars frame.

> **This is the make-or-break correctness check.** The retrieved chunks must be *on-topic for the
> question*. If they are about something unrelated, the query embedding is wrong (wrong model,
> wrong normalization, a stray prefix) — and no amount of clever prompting downstream will save an
> answer grounded in the wrong context. Read the table before trusting the answer.

In [4]:
TOP_K = 5
hits = qdrant.query_points(COLLECTION, query=query_vec, limit=TOP_K, with_payload=True).points

retrieved = pl.DataFrame([
    {"rank": i,
     "score": round(h.score, 4),
     "source_name": h.payload.get("source_name"),
     "chunk_title": h.payload.get("chunk_title"),
     "snippet": snippet(h.payload.get("content"))}
    for i, h in enumerate(hits, 1)
])
print(f"top {TOP_K} chunks for: {question!r}\n")
retrieved

top 5 chunks for: 'How does the lab do GitOps deployment?'



rank,score,source_name,chunk_title,snippet
i64,f64,str,str,str
1,0.7539,"""ci-cd.md""","""In Practice""","""## In Practice the team uses G…"
2,0.7216,"""gitops.md""","""In Practice""","""## In Practice GitOps is our r…"
3,0.7181,"""gitops.md""","""Related Entries""","""## Related Entries - [Infrastr…"
4,0.7033,"""gitops.md""","""When to Apply""","""## When to Apply - Kubernetes-…"
5,0.7031,"""gitops.md""","""Engineering Knowledge""","""## Engineering Knowledge 💡 **E…"


Read the frame top-down: every row should be about **how the lab deploys — GitOps, Argo,
push-to-git, sync** — drawn from the lab's own docs (`arch.md`, `README.md`, backlog, runbooks).
That on-topic set is the proof the query landed in the right vector space; these are the exact
passages we now hand the model as context. If instead you saw music-dataset chunks or unrelated
config, you would stop here and fix the embedding — not proceed to generation.

## Augment + generate — a grounded answer

This is the *augment* and *generate* half. We build a prompt in two parts:

- a **system** message that pins the model's job: answer **only** from the supplied context, and
  say so plainly when the context doesn't cover the question (this is what stops it from falling
  back on its parametric guesses — the whole point of grounding);
- a **user** message that carries the question **plus the retrieved chunks** as labelled context.

Then we send it to the LiteLLM gateway's `wl-rag` lane and print the answer. Because the context
is real lab documentation, the answer should describe *this* lab's actual GitOps setup — not
GitOps in the abstract.

In [5]:
# stitch the retrieved chunks into a labelled context block (source + text) the model can cite
context = "\n\n".join(
    f"[{i}] source: {h.payload.get('source_name')}\n{h.payload.get('content')}"
    for i, h in enumerate(hits, 1)
)

SYSTEM = (
    "You are a precise assistant for the weyland homelab platform. Answer the question using ONLY "
    "the numbered context passages provided. Cite the sources you use by their [n] marker. If the "
    "context does not contain the answer, say so plainly instead of guessing."
)
USER = f"Context:\n{context}\n\nQuestion: {question}"

resp = llm.chat.completions.create(
    model=GEN_MODEL,
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user", "content": USER}],
    temperature=0.1,
)
grounded_answer = (resp.choices[0].message.content or "").strip()
print("GROUNDED answer (wl-rag, retrieved context in the prompt):\n")
print(grounded_answer)

GROUNDED answer (wl-rag, retrieved context in the prompt):

The lab implements GitOps by using **Argo CD** as the deployment engine for Kubernetes.  
- All manifests live in Git (often split into separate directories or repos per environment).  
- A merge of a pull request triggers Argo CD to reconcile the cluster with the desired state, providing UI diff views and automated sync.  
- Secrets are kept out of plain‑text Git by using Sealed Secrets or an External Secrets Operator.  
- Production releases are promoted manually from staging (a manual promotion step is required).  

[1] [2]


## Contrast — the same question with no context

To see what retrieval actually bought, ask the **same model the same question with no context at
all** — just the bare question. The model can only answer from its training data, which never
included this lab. Expect one of two failure modes: a generic essay about GitOps-in-general
(plausible but not about *this* lab), or an honest "I don't have specifics." Either way it lacks
the concrete, lab-specific detail the grounded answer carries — that difference **is** the value
RAG adds.

In [6]:
resp_raw = llm.chat.completions.create(
    model=GEN_MODEL,
    messages=[{"role": "user", "content": question}],
    temperature=0.1,
)
raw_answer = (resp_raw.choices[0].message.content or "").strip()
print("NO-CONTEXT answer (wl-rag, question only - the model's parametric guess):\n")
print(raw_answer)

NO-CONTEXT answer (wl-rag, question only - the model's parametric guess):

## What is “GitOps” in a nutshell?

GitOps turns **Git** into the single source of truth for your entire deployment pipeline:

| Layer | Traditional CI/CD | GitOps |
|-------|-------------------|--------|
| **Source of truth** | Code + build artifacts (often stored in an artifact repo) | *Everything* – code, container images, Kubernetes manifests, Helm charts, Kustomize overlays, etc. – lives in a Git repository. |
| **Change flow** | Commit → CI builds → Artifact push → CD deploys via scripts or pipelines | Commit → CI builds → Push image → Update manifest (or tag) → Commit back to repo → CD tool watches the repo and applies changes automatically. |
| **Observability & audit** | Logs from CI/CD jobs | Git history + pull‑request reviews + automated reconciliation logs. |

In a lab setting, you’ll usually see one of two popular GitOps tools in action:

* **Argo CD** – a declarative, Git‑backed continuous delivery

## When to reach for RAG — and the pieces that build it

**Reach for RAG when the answer lives in a corpus the model was never trained on** — your own
docs, code, runbooks, tickets, or any body of text that changes faster than a model is retrained.
It grounds the model in retrieved facts, cuts hallucination on domain questions, lets you cite
sources, and updates the moment the corpus does — no fine-tuning, no retraining. The grounded vs.
no-context contrast above is the whole case in one screen.

**Reach for something else when RAG doesn't fit.** If the task needs *reasoning* rather than
*recall* (math, planning, code transformation), retrieval adds noise — use the model directly
(`wl-reason`). If the knowledge is small and stable, stuffing it straight into the system prompt
beats standing up a retriever. And if answers must be perfectly authoritative, RAG narrows but
never eliminates the model's freedom to misread its context — pair it with evaluation.

**The three pieces, and where each one lives in this library:**

| piece | its job | where in this library |
|-------|---------|------------------------|
| **embedder** (`bge-base-en-v1.5`) | turn text — corpus *and* query — into comparable vectors | loaded here; the corpus was built by the hydration pipeline |
| **vector store** (Qdrant) | hold the corpus vectors, return nearest neighbours fast | notebook `30` — discovery, filtered ANN, tuning |
| **LLM gateway** (LiteLLM) | generate the grounded answer from question + context | notebook `61` — the gateways, lanes, and fallbacks |

> **RAG is not a model — it's a pipeline.** Embed with the corpus's model, retrieve the nearest
> chunks, augment the prompt, generate. Get the *embedder parity* right (the one rule at the top)
> and the retrieval is sound; get the *grounding prompt* right and the generation stays honest.
> This notebook is the smallest end-to-end version of that loop, run against the live mesh.